<div dir="rtl">
<h1>یک Token در دو جای متفاوت</h1>
<p>درس 30 از 76 · چرا جای Token را هم نمایش می‌دهیم؟ · <code dir="ltr">26-positions</code></p>
<p><a target="_self" href="http://127.0.0.1:8000/part-04/chapter-03/26-positions.html">📖 بازگشت به همین درس</a></p>
<p>نمایش Token و Position را با حفظ محورهای Batch، زمان و ویژگی ترکیب کنید.</p><p>پیش‌نیاز: 15-broadcast و25-embedding؛ هنوز ساخت Attention لازم نیست.</p>
<p>این دفتر نیمهٔ عملی درس است. مثال‌ها آمادهٔ اجرا هستند؛ دو Cell با برچسب TODO را خودتان کامل کنید. پیام INCOMPLETE یعنی هنوز چیزی ننوشته‌اید، نه اینکه پاسخ درست است. جواب مرجع در این دفتر پنهان نشده است.</p>
<p>از بالا به پایین اجرا کنید. پس از تغییر هر تابع، Cell آن و سپس Cell آزمون را دوباره اجرا کنید. برای بررسی نهایی، از منوی <code>Kernel → Restart Kernel and Run All Cells</code> استفاده کنید.</p>
</div>

In [ ]:
from pathlib import Path
import os
import sys

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "mini_gpt").is_dir() and (p / "book_src").is_dir()), None)
if project_root is None:
    raise RuntimeError("Extract the complete learning project; open this notebook inside it.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Python:", sys.executable)
print("Project:", project_root)

<div dir="rtl">
<h2>قبل از اجرا، پیش‌بینی کنید</h2>
<p>دو وقوع ID یک، قبل و بعد از افزودن موقعیت چه رابطه‌ای دارند؟ آیا جمع دو Vector Cتایی خروجی 2Cتایی می‌دهد؟</p>
</div>

<div dir="rtl"><p>پیش‌بینی من: …</p></div>

In [ ]:
import torch
from torch import nn
token_table = torch.tensor([[0., 0.], [2., 3.], [4., 5.]])
position_table = torch.tensor([[0., 1.], [1., 0.], [-1., 2.], [2., -1.]])
ids = torch.tensor([[1, 2, 1]], dtype=torch.long)
print("Same token at positions 0 and 2:", token_table[ids])

<div dir="rtl">
<h2>این بار شما کد بنویسید</h2>
<p>تابع add_positions(ids,token_table,position_table) سطرهای Token را بردارد و برای موقعیت t سطر t از جدول Position را جمع کند. موقعیت‌ها برای هر نمونه از صفر شروع می‌شوند. طول بیش از تعداد سطر Position را با ValueError رد کنید.</p>
</div>

In [ ]:
def add_positions(ids, token_table, position_table):
    # TODO: add position vectors across the batch axis
    return None

In [ ]:
def test_exercise():
    result = add_positions(ids, token_table, position_table)
    if result is None:
        return False
    assert tuple(result.shape) == (1, 3, 2)
    assert result.tolist() == [[[2., 4.], [5., 5.], [1., 5.]]]
    two = add_positions(ids.repeat(2, 1), token_table, position_table)
    assert torch.equal(two[0], two[1])
    try:
        add_positions(torch.ones(1, 5, dtype=torch.long), token_table, position_table)
    except ValueError:
        pass
    else:
        raise AssertionError("Position table has a finite context limit")
    return True

exercise_complete = test_exercise()
print('PASS' if exercise_complete else 'INCOMPLETE: implement the TODO and rerun')

<div dir="rtl">
<h2>فقط یک عامل را تغییر دهید</h2>
<p>فقط جدول Position را صفر کنید؛ Tokenها و IDها ثابت‌اند. برابری دو وقوع یک ID باید برگردد. این مشاهده دربارهٔ همین مرحله است، نه تمام اطلاعات ترتیب در معماری کامل.</p>
</div>

In [ ]:
base = token_table[ids]
for positions in [position_table, torch.zeros_like(position_table)]:
    shown = base+positions[:ids.shape[1]]
    print("Position 0:", shown[0, 0].tolist(), "Position 2:", shown[0, 2].tolist())

<div dir="rtl">
<h2>خرابی را پیدا کنید</h2>
<p>در نسخهٔ خراب، ID Token به‌جای شمارهٔ موقعیت به جدول Position داده می‌شود. تابع positions_for(ids) Tensor شماره‌های ۰ تا T-1 را روی دستگاه ids برگرداند؛ شکل خروجی (T,) است.</p>
</div>

In [ ]:
wrong = token_table[ids]+position_table[ids]
print("Broken repeated-token equality:", torch.equal(wrong[0, 0], wrong[0, 2]))
assert torch.equal(wrong[0, 0], wrong[0, 2])

<div dir="rtl">
<h2>اصلاح را خودتان بنویسید</h2>
<p>علت را توضیح دهید، سپس تابع زیر را کامل کنید. خطای عمدی بالا یک نمونهٔ آموزشی است؛ آزمون پایین باید اصلاح شما را بسنجد.</p>
</div>

In [ ]:
def positions_for(ids):
    # TODO: depend on sequence length, not on token values
    return None

In [ ]:
def test_repair():
    result = positions_for(ids)
    if result is None:
        return False
    assert result.tolist() == [0, 1, 2]
    assert result.dtype == torch.long and result.device == ids.device
    assert positions_for(torch.tensor([[2, 2]])).tolist() == [0, 1]
    return True

repair_complete = test_repair()
print('PASS' if repair_complete else 'INCOMPLETE: implement the TODO and rerun')

<div dir="rtl">
<h2>در Mini-GPT کجا به کار می‌آید؟</h2>
<p>mini_gpt/stages/v4.py و MiniGPT.forward همین انتخاب موقعیت با arange و جمع دو جدول را انجام می‌دهند. اینجا آن جزء را بدون اجرای Attention جدا کردیم.</p>
</div>

<div dir="rtl">
<h2>با زبان خودتان توضیح دهید</h2>
<p>چرا متفاوت‌شدن دو بردار، هنوز شاهد یادگرفتن معنای ترتیب نیست؟</p>
</div>
<div dir="rtl"><p>پیش‌بینی و مشاهدهٔ من: …</p><p>علت خرابی و اصلاح من: …</p></div>

<div dir="rtl"><p><a target="_self" href="http://127.0.0.1:8000/part-04/chapter-03/26-positions.html">بازگشت به درس و ادامهٔ مسیر</a> · <a target="_self" href="http://127.0.0.1:8000/answers/26-positions.html#lab-solution">فقط پس از تلاش: راه‌حل مرجع آزمایشگاه</a></p></div>